In [ ]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report,
    confusion_matrix, accuracy_score, f1_score)
from sklearn.model_selection import train_test_split

# Load everything
df = pd.read_csv("../data/processed/grievances_cleaned.csv")

# Re-create sentiment labels (same logic as notebook 05)
import random; random.seed(42)
urgent_kw  = ["live wire","sparks","sparking","fallen","burst",
              "dangerous","burning","accident","cave","contamination",
              "overflow","sewage","mixing"]
negative_kw = ["not working","no water","no electricity","not coming",
               "broken","poor","wrong","overcharging","illegal",
               "blocked","flooded","encroached"]

def assign_s(t):
    tl=t.lower()
    for k in urgent_kw:
        if k in tl: return "Urgent/Critical"
    for k in negative_kw:
        if k in tl: return "Negative"
    return random.choice(["Neutral","Negative"])

df["sentiment"] = df["complaint_text"].apply(assign_s)

# Load all 4 model files
with open("../src/model.pkl","rb") as f: dept_model=pickle.load(f)
with open("../src/vectorizer.pkl","rb") as f: dept_vec=pickle.load(f)
with open("../src/sentiment_model.pkl","rb") as f: sent_model=pickle.load(f)
with open("../src/sentiment_vectorizer.pkl","rb") as f: sent_vec=pickle.load(f)

# Department evaluation
Xd=df["cleaned_text"]; yd=df["department"]
_,Xd_t,_,yd_t=train_test_split(Xd,yd,test_size=0.2,random_state=42,stratify=yd)
yd_pred=dept_model.predict(dept_vec.transform(Xd_t))
print("=== DEPARTMENT MODEL ===")
print(f"Accuracy: {accuracy_score(yd_t,yd_pred):.4f}")
print(f"Macro F1: {f1_score(yd_t,yd_pred,average='macro'):.4f}")
print(classification_report(yd_t,yd_pred))

# Sentiment evaluation
Xs=df["cleaned_text"]; ys=df["sentiment"]
_,Xs_t,_,ys_t=train_test_split(Xs,ys,test_size=0.2,random_state=42,stratify=ys)
ys_pred=sent_model.predict(sent_vec.transform(Xs_t))
print("=== SENTIMENT MODEL ===")
print(f"Accuracy: {accuracy_score(ys_t,ys_pred):.4f}")
print(f"Macro F1: {f1_score(ys_t,ys_pred,average='macro'):.4f}")
print(classification_report(ys_t,ys_pred)) 

=== DEPARTMENT MODEL ===
Accuracy: 0.6500
Macro F1: 0.6407
              precision    recall  f1-score   support

 Electricity       0.67      0.50      0.57         4
       Roads       0.57      1.00      0.73         4
  Sanitation       0.67      0.50      0.57         4
   Transport       1.00      0.50      0.67         4
       Water       0.60      0.75      0.67         4

    accuracy                           0.65        20
   macro avg       0.70      0.65      0.64        20
weighted avg       0.70      0.65      0.64        20

=== SENTIMENT MODEL ===
Accuracy: 0.7500
Macro F1: 0.7760
                 precision    recall  f1-score   support

       Negative       0.69      0.90      0.78        10
        Neutral       0.75      0.43      0.55         7
Urgent/Critical       1.00      1.00      1.00         3

       accuracy                           0.75        20
      macro avg       0.81      0.78      0.78        20
   weighted avg       0.76      0.75      0.73    